# Zero-shot Tutorial

- Creators: TERRA team
- Date of Creation: 30.06.2025
- Date of Last Modification: 21.06.2026

Run this notebook in the `terra` environment (install `terra` and its dependencies first).

To accelerate clustering, you can optionally run section **2.6** in an environment with `rapids-singlecell` installed.

## 1. Setup

### 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

In [3]:
import gc
import logging
import os
import pickle
import warnings
from typing import List, Tuple
#from tqdm import tqdm

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import scipy.sparse as sp
try:
    import squidpy as sq
except:
    print("Could not import squidpy")
    import rapids_singlecell as rsc
import torch
from datasets import concatenate_datasets, load_from_disk
from itertools import combinations
from matplotlib.colors import Normalize
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

from terra import (embed_dataset,
                           harmonize_adata,
                           perturb_dataset,
                           tokenize_adata,
                           harmonize_tokenize_embed_pipeline,
                           get_gene_embed,
                           get_average_gene_embed,
                           get_spatial_score,
                           get_emd_distance)
from terra.utils.helper import init_model, load_checkpoint
from terra.utils.evaluation import (get_top_gene_score,
                                        get_top_gene_pairs)

# The following are optional local helper modules and are NOT part of `terra`.
# They are only used for a few benchmarking plots in this tutorial; uncomment if
# you have them available locally.
#from analysis_utils import *
#from benchmarking_utils import max_overlap_plot

# Make pipeline progress visible in the notebook
logging.basicConfig(level="INFO")

/path/to/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/path/to/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/path/to/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


### 1.2 Configure Paths

**TODO**: Specify timestamp of model checkpoint.

In [4]:
from terra import download_pretrained

# Downloads the pretrained TERRA model bundle from the Hugging Face Hub and
# returns the LOCAL folder path to pass as `model_folder_path` downstream.
model_folder_path = download_pretrained("Lotfollahi-lab/TERRA-96M")

# These labels namespace the output folders created downstream.
model_timestamp = "TERRA-96M"
model_checkpoint = "ep0"

**TODO**: Adjust `artifacts_folder_path` to a location where you want store outputs.

In [5]:
# Local folder where tutorial outputs (results, caches, harmonized/tokenized/embedded data) are stored.
artifacts_folder_path = "./terra_tutorial_artifacts"

In [6]:
results_folder_path = f'{artifacts_folder_path}/results/{model_timestamp}'
cache_directory_path = f'{artifacts_folder_path}/cache'
harmonized_data_folder_path = f'{artifacts_folder_path}/harmonized_data'
tokenized_data_folder_path = f'{artifacts_folder_path}/tokenized_data'
embedded_data_folder_path = f'{artifacts_folder_path}/embedded_data'

### 1.3 Define Config

**TODO**: To provide your input data, you have three different options. Please pick the correct option for your case and set the config in the following notebook cell accordingly.
- Option 1: specify a single file (`file_path`) containing an AnnData object with a single section.
    - In this case, set `folder_path` and `sample_key` to None.
- Option 2: specify a folder (`folder_path`) with multiple AnnData objects (one per section).
    - In this case, set `file_path` and `sample_key` to None.
- Option 3: specify a file (`file_path`) containing an AnnData object with multiple sections.
    - In this case, set `folder_path` to None and specify the `sample_key` as the adata.obs column that is unique for each section.
 
For all options, also define a project name where outputs will be saved.

In [8]:
########################################
### EDIT ME: configure your input data ###
########################################
# Provide a single .h5ad file via `file_path`, or a folder of .h5ad files via
# `folder_path`. Set `sample_key` to an `adata.obs` column name to split a single
# file into multiple samples, otherwise leave it as None.
#
# Keep these four variable names as-is: downstream cells rely on them.
project_name = "my_terra_run"
file_path = "/path/to/your_spatial_data.h5ad"
sample_key = None
folder_path = None

**TODO**: Specify other dataset-specific parameters.

In [9]:
label_keys = ['total_counts']

**TODO**: Specify inference and analysis parameters.

In [10]:
# Harmonization params
min_cells_per_gene = 0
min_genes_per_cell = 10

# Tokenization params
nproc = 4
processing_mode = 'parallel'
use_generator = True
num_shards = 32
ensembl_release = 110
species = "human"
# Auto-resolved from the model bundle (`model_folder_path`); leave as None.
gene_mapping_dict_file_path = None
gene_occurrence_count_file_path = None
gene_occurrence_count_filter_value = 10

# Model params
emb_layer = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_genes = None
top_k = None

# Data params
batch_key = 'batch'
n_pca_pcs = 20

# Clustering params
n_pcs = None
latent_leiden_resolution = 0.6

### 1.4 Run Notebook Setup

In [11]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [12]:
# Set plotting parameters
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

sc.settings.figdir = f'{results_folder_path}/{project_name}'
sc.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [13]:
# Create folders
os.makedirs(results_folder_path, exist_ok=True)
os.makedirs(f'{results_folder_path}/{project_name}', exist_ok=True)
os.makedirs(harmonized_data_folder_path, exist_ok=True)
os.makedirs(tokenized_data_folder_path, exist_ok=True)
os.makedirs(embedded_data_folder_path, exist_ok=True)
os.makedirs(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}", exist_ok=True)
os.makedirs(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}", exist_ok=True)
os.makedirs(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}", exist_ok=True)

### 1.5 Define Functions

In [14]:
def subset_by_cell_ids(dataset, cell_id_list):
    """
    Takes a huggingface dataset and a list of cell ids and returns the corresponding filtered dataset object.
    """
    cell_id_set = set(cell_id_list)
    cell_ids = dataset['cell_id']
    indices = [i for i, cid in tqdm(enumerate(cell_ids), total=len(cell_ids), desc="Finding matching indices") if cid in cell_id_set]
    return dataset.select(indices)

In [15]:
def summarize_gene_expression(
    adata,
    subset_cell_ids: list,
    top_n: int = 10,
    ensembl_col: str = "ensembl_id"
):
    """
    Filter AnnData to a subset of cells, compute gene expression summaries,
    and return top expressed genes with Ensembl IDs.

    Args:
        adata (AnnData): Pre-loaded AnnData object.
        subset_cell_ids (list): List of cell IDs to filter.
        top_n (int): Number of top expressed genes to return.
        ensembl_col (str): Column in adata.var that contains Ensembl IDs.

    Returns:
        combined_counts (pd.DataFrame): Filtered cell-by-gene count matrix.
        top_gene_df (pd.DataFrame): DataFrame with top gene names and Ensembl IDs.
    """
    if 'cell_id' not in adata.obs.columns:
        raise KeyError("Expected `adata.obs` to contain a 'cell_id' column.")
    if ensembl_col not in adata.var.columns:
        raise KeyError(f"Expected `adata.var` to contain '{ensembl_col}' column.")

    # Filter to subset of cells
    mask = adata.obs['cell_id'].astype(str).isin(set(subset_cell_ids))
    filtered = adata[mask].copy()
    if filtered.n_obs == 0:
        raise ValueError("No matching cells found in the provided subset.")

    # Extract dense matrix
    matrix = filtered.X.toarray() if hasattr(filtered.X, "toarray") else filtered.X
    combined_counts = pd.DataFrame(matrix, columns=filtered.var_names, index=filtered.obs_names)

    # Compute mean expression and get top N genes
    mean_expr = combined_counts.mean(axis=0)
    top_genes = mean_expr.nlargest(top_n)

    # Map to Ensembl IDs
    var_df = adata.var.loc[top_genes.index]
    top_gene_df = pd.DataFrame({
        "gene_name": top_genes.index,
        "ensembl_id": var_df[ensembl_col].values,
        "mean_expression": top_genes.values
    })

    return combined_counts, top_gene_df

## 2. Analysis

### 2.1 (Optional) Visualize Data

In [ ]:
if folder_path: # Option 2
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".h5ad"):
            file_path = os.path.join(folder_path, file_name)
            print(f"Reading {file_path}.")
    
            adata = sc.read_h5ad(file_path)
            spot_sizes = {key: 20 for key in adata.obs[sample_key].unique().tolist()}
            print(adata)
            if 'spatial' in adata.uns.keys():
                del(adata.uns['spatial'])

            for label_key in label_keys:
                sc.pl.spatial(adata,
                              spot_size=spot_sizes[file_name.replace('.h5ad', '')],
                              color=label_key)
else:
    adata = sc.read_h5ad(file_path)
    spot_sizes = {key: 20 for key in adata.obs[sample_key].unique().tolist()}
    print(adata)
    if 'spatial' in adata.uns.keys():
        del(adata.uns['spatial'])

    if sample_key: # Option 3
        for sample in adata.obs[sample_key].unique().tolist():
            adata_sample = adata[adata.obs[sample_key] == sample]
            for label_key in label_keys:
                sc.pl.spatial(adata_sample,
                              spot_size=spot_sizes[sample],
                              color=label_key) 
    else: # Option 1
        for label_key in label_keys:
            sc.pl.spatial(adata,
                          spot_size=spot_sizes[file_path.split('/')[-1].replace('.h5ad', '')],
                          color=label_key) 

### 2.2 Harmonize, Tokenize & Embed Data to Retrieve Global Embeddings

**TODO**: To retrieve TERRA embeddings, the input AnnData object first needs to be harmonized and tokenized. You can either run the end-to-end pipeline from step **2.2.1** OR the step-by-step approach from **2.2.2**.

#### 2.2.1 End-to-end Pipeline

**TODO**: Skip this if you want to run the step-by-step-approach in **2.2.2**.

In [ ]:
if folder_path:
    adata_list = []
    for i, file_name in enumerate(os.listdir(folder_path)):
        if file_name.endswith(".h5ad") or file_name.endswith(".h5ad.gz"):
            file_path = os.path.join(folder_path, file_name)
            sample_name = file_name.replace(".h5ad.gz", ".h5ad")
            sample_name = sample_name.replace(".h5ad", "")
    
            adata_sample = sc.read_h5ad(file_path)

            adata_sample = harmonize_tokenize_embed_pipeline(
                adata=adata_sample,
                sample_key=sample_key,
                batch_key=batch_key,
                model_folder_path=model_folder_path,
                cache_directory_path=cache_directory_path,
                gene_mapping_dict_file_path=gene_mapping_dict_file_path,
                gene_occurrence_count_file_path=gene_occurrence_count_file_path,
                gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
                ensembl_release=ensembl_release,
                species=species,
                gene_perturb_df=None,               
                nproc=nproc,
                processing_mode=processing_mode,
                harmonized_adata_save_path=f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad",
                save_dataset_path=f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
                num_shards=num_shards,
                emb_layer=emb_layer,
                agg_excluded_genes=agg_excluded_genes,
                top_k=top_k,
                batch_size=batch_size,
                pin_memory=pin_memory,
                num_workers=num_workers,
                use_generator=use_generator,
                add_neigh_cell_ids=False,
                min_cells_per_gene=min_cells_per_gene,
                min_genes_per_cell=min_genes_per_cell)

            adata_sample.obs[batch_key] = sample_name
            adata_sample.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}_{model_checkpoint}.h5ad")
            adata_list.append(adata_sample)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
else:
    file_name = file_path.split('/')[-1]
    sample_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = sample_name.replace(".h5ad", "")

    adata = sc.read_h5ad(file_path)
    
    adata = harmonize_tokenize_embed_pipeline(
        adata=adata,
        sample_key=sample_key,
        batch_key=batch_key,
        model_folder_path=model_folder_path,
        cache_directory_path=cache_directory_path,
        gene_mapping_dict_file_path=gene_mapping_dict_file_path,
        gene_occurrence_count_file_path=gene_occurrence_count_file_path,
        gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
        ensembl_release=ensembl_release,
        species=species,
        gene_perturb_df=None,               
        nproc=nproc,
        processing_mode=processing_mode,
        harmonized_adata_save_path=f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad",
        save_dataset_path=f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
        num_shards=num_shards,
        emb_layer=emb_layer,
        agg_excluded_genes=agg_excluded_genes,
        top_k=top_k,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        use_generator=use_generator,
        add_neigh_cell_ids=False,
        min_cells_per_gene=min_cells_per_gene,
        min_genes_per_cell=min_genes_per_cell)

    adata.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}_{model_checkpoint}.h5ad")

#### 2.2.2 Step-by-Step Approach

**TODO**: Skip this if you ran **2.2.1 End-to-end Pipeline**.

##### 2.2.2.1 Harmonize Data

**TODO**: Skip this and proceed to **2.2.2.3 Load Tokenized Data** if you already harmonized and tokenized your data and saved the results.

In [16]:
if folder_path:
    adata_list = []
    for i, file_name in enumerate(os.listdir(folder_path)):
        if file_name.endswith(".h5ad") or file_name.endswith(".h5ad.gz"):
            file_path = os.path.join(folder_path, file_name)
            sample_name = file_name.replace(".h5ad.gz", ".h5ad")
            sample_name = sample_name.replace(".h5ad", "")
            
            adata_sample = sc.read_h5ad(file_path)

            adata_sample.obs_names_make_unique()

            adata_sample = harmonize_adata(
                       adata_sample,
                       gene_mapping_dict_file_path=gene_mapping_dict_file_path,
                       gene_occurrence_count_file_path=gene_occurrence_count_file_path,
                       gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
                       ensembl_release=ensembl_release,
                       species=species,
                       min_cells_per_gene=min_cells_per_gene,
                       min_genes_per_cell=min_genes_per_cell)
            adata_sample.obs[batch_key] = sample_name
            adata_sample.write(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
            adata_list.append(adata_sample)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
elif sample_key:
    adata = sc.read_h5ad(file_path)
    adata_list = []
    samples = adata.obs[sample_key].unique().tolist()
    for sample_name in samples:
        adata_sample = adata[adata.obs[sample_key] == sample_name]

        adata_sample.obs_names_make_unique()

        adata_sample = harmonize_adata(
            adata_sample,
            gene_mapping_dict_file_path=gene_mapping_dict_file_path,
            gene_occurrence_count_file_path=gene_occurrence_count_file_path,
            gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
            ensembl_release=ensembl_release,
            species=species,
            min_cells_per_gene=min_cells_per_gene,
            min_genes_per_cell=min_genes_per_cell)
        adata_sample.obs[batch_key] = sample_name
        adata_sample.write(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
        adata_list.append(adata_sample)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
else:
    file_name = file_path.split('/')[-1]
    sample_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = sample_name.replace(".h5ad", "")
    adata = sc.read_h5ad(file_path)
    adata = harmonize_adata(
        adata,
        gene_mapping_dict_file_path=gene_mapping_dict_file_path,
        gene_occurrence_count_file_path=gene_occurrence_count_file_path,
        gene_occurrence_count_filter_value=gene_occurrence_count_filter_value,
        ensembl_release=ensembl_release,
        species=species,
        min_cells_per_gene=min_cells_per_gene,
        min_genes_per_cell=min_genes_per_cell)
    adata.obs[batch_key] = sample_name
    adata.write(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
print(adata)

STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
✓ adata.X contains raw counts (integer values).
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from gene_mapping_dict with file path `/path/to/gene_name_to_ensembl_id_dict.pkl`...
Number of genes with matching ensembl IDs: 4949.
Number of genes skipped due to non-matching ensembl IDs: 52.
Genes excluded due to non-matching ensembl IDs: {'EPRS', 'H3F3B', 'ZBED9', 'MIAT', 'TRDC', 'PCAT1', 'WARS', 'PVT1', 'IGHE', 'MEG3', 'DLEU1', 'HSPB11', 'SNHG1', 'ACPP', 'CYHR1', 'SNHG11', 'PCA3', 'CYTOR', 'HOTAIR', 'PCGEM1', 'CASC8', 'DLEU2', 'FAM126B', 'NORAD', 'H2AFX', 'CBWD2', 'ARNTL', 'PHB', 'DDX58', 'PART1', 'SLC9A3R1', 'CBSL', 'SPATA5L1', 'SNHG15', 'PRNCR1', 'GBA', 'GPR1', 'SOX2-OT', 'UCA1', 'TMEM173', 'CD3EAP', 'FENDRR', 'DUSP13', 'SPATA5', 'SNHG8', 'EEF1AKNMT', 'BCAR4', 'SLC9A3R2', 'H19', 'CCAT2', 'CRNDE', 'SNHG17'}.
Filtering genes that have not occurred enough during pretraining...
Number of genes skipped due to not eno

... storing 'batch' as categorical


Before gene filtering: 4949 genes.
After gene filtering: 4949 genes (removed 0 genes).
STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
✓ adata.X contains raw counts (integer values).
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from gene_mapping_dict with file path `/path/to/gene_name_to_ensembl_id_dict.pkl`...
Number of genes with matching ensembl IDs: 4949.
Number of genes skipped due to non-matching ensembl IDs: 52.
Genes excluded due to non-matching ensembl IDs: {'EPRS', 'H3F3B', 'ZBED9', 'MIAT', 'TRDC', 'PCAT1', 'WARS', 'PVT1', 'IGHE', 'MEG3', 'DLEU1', 'HSPB11', 'SNHG1', 'ACPP', 'CYHR1', 'SNHG11', 'PCA3', 'CYTOR', 'HOTAIR', 'PCGEM1', 'CASC8', 'DLEU2', 'FAM126B', 'NORAD', 'H2AFX', 'CBWD2', 'ARNTL', 'PHB', 'DDX58', 'PART1', 'SLC9A3R1', 'CBSL', 'SPATA5L1', 'SNHG15', 'PRNCR1', 'GBA', 'GPR1', 'SOX2-OT', 'UCA1', 'TMEM173', 'CD3EAP', 'FENDRR', 'DUSP13', 'SPATA5', 'SNHG8', 'EEF1AKNMT', 'BCAR4', 'SLC9A3R2', 'H19', 'CCAT2', 'CRNDE', 'SNHG17'}.
Filtering genes tha

... storing 'batch' as categorical


Before gene filtering: 4949 genes.
After gene filtering: 4949 genes (removed 0 genes).
STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
✓ adata.X contains raw counts (integer values).
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from gene_mapping_dict with file path `/path/to/gene_name_to_ensembl_id_dict.pkl`...
Number of genes with matching ensembl IDs: 4949.
Number of genes skipped due to non-matching ensembl IDs: 52.
Genes excluded due to non-matching ensembl IDs: {'EPRS', 'H3F3B', 'ZBED9', 'MIAT', 'TRDC', 'PCAT1', 'WARS', 'PVT1', 'IGHE', 'MEG3', 'DLEU1', 'HSPB11', 'SNHG1', 'ACPP', 'CYHR1', 'SNHG11', 'PCA3', 'CYTOR', 'HOTAIR', 'PCGEM1', 'CASC8', 'DLEU2', 'FAM126B', 'NORAD', 'H2AFX', 'CBWD2', 'ARNTL', 'PHB', 'DDX58', 'PART1', 'SLC9A3R1', 'CBSL', 'SPATA5L1', 'SNHG15', 'PRNCR1', 'GBA', 'GPR1', 'SOX2-OT', 'UCA1', 'TMEM173', 'CD3EAP', 'FENDRR', 'DUSP13', 'SPATA5', 'SNHG8', 'EEF1AKNMT', 'BCAR4', 'SLC9A3R2', 'H19', 'CCAT2', 'CRNDE', 'SNHG17'}.
Filtering genes tha

...

Before gene filtering: 4949 genes.
After gene filtering: 4949 genes (removed 0 genes).
STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
✓ adata.X contains raw counts (integer values).
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from gene_mapping_dict with file path `/path/to/gene_name_to_ensembl_id_dict.pkl`...
Number of genes with matching ensembl IDs: 4949.
Number of genes skipped due to non-matching ensembl IDs: 52.
Genes excluded due to non-matching ensembl IDs: {'EPRS', 'H3F3B', 'ZBED9', 'MIAT', 'TRDC', 'PCAT1', 'WARS', 'PVT1', 'IGHE', 'MEG3', 'DLEU1', 'HSPB11', 'SNHG1', 'ACPP', 'CYHR1', 'SNHG11', 'PCA3', 'CYTOR', 'HOTAIR', 'PCGEM1', 'CASC8', 'DLEU2', 'FAM126B', 'NORAD', 'H2AFX', 'CBWD2', 'ARNTL', 'PHB', 'DDX58', 'PART1', 'SLC9A3R1', 'CBSL', 'SPATA5L1', 'SNHG15', 'PRNCR1', 'GBA', 'GPR1', 'SOX2-OT', 'UCA1', 'TMEM173', 'CD3EAP', 'FENDRR', 'DUSP13', 'SPATA5', 'SNHG8', 'EEF1AKNMT', 'BCAR4', 'SLC9A3R2', 'H19', 'CCAT2', 'CRNDE', 'SNHG17'}.
Filtering genes tha

... storing 'batch' as categorical


Before gene filtering: 4949 genes.
After gene filtering: 4949 genes (removed 0 genes).
STEP 1: DATA VALIDATION...
Checking that adata.X contains raw counts...
✓ adata.X contains raw counts (integer values).
STEP 2: ADDING ENSEMBL IDS...
Adding ensembl IDs from gene_mapping_dict with file path `/path/to/gene_name_to_ensembl_id_dict.pkl`...
Number of genes with matching ensembl IDs: 4949.
Number of genes skipped due to non-matching ensembl IDs: 52.
Genes excluded due to non-matching ensembl IDs: {'EPRS', 'H3F3B', 'ZBED9', 'MIAT', 'TRDC', 'PCAT1', 'WARS', 'PVT1', 'IGHE', 'MEG3', 'DLEU1', 'HSPB11', 'SNHG1', 'ACPP', 'CYHR1', 'SNHG11', 'PCA3', 'CYTOR', 'HOTAIR', 'PCGEM1', 'CASC8', 'DLEU2', 'FAM126B', 'NORAD', 'H2AFX', 'CBWD2', 'ARNTL', 'PHB', 'DDX58', 'PART1', 'SLC9A3R1', 'CBSL', 'SPATA5L1', 'SNHG15', 'PRNCR1', 'GBA', 'GPR1', 'SOX2-OT', 'UCA1', 'TMEM173', 'CD3EAP', 'FENDRR', 'DUSP13', 'SPATA5', 'SNHG8', 'EEF1AKNMT', 'BCAR4', 'SLC9A3R2', 'H19', 'CCAT2', 'CRNDE', 'SNHG17'}.
Filtering genes tha

... storing 'batch' as categorical


Before gene filtering: 4949 genes.
After gene filtering: 4949 genes (removed 0 genes).
AnnData object with n_obs × n_vars = 4393579 × 4949
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample_id2', 'run_id', 'sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'n_counts', 'info_id2', 'dataset', 'sequencing_lane_ID', 'haniffa_ID', 'HDBR_ID', 'sequencing_type', '10x_kit', 'spatial_location', 'spatial_location_replicates', 'sort_ID', 'procedure', 'age_in_cs', 'sex', 'alignment_software', 'alignment_reference', 'nGene', 'nGene_eq-to-morethan_500', 'nUMI', 'nUMI_morethan_1000', 'passes_QC_step_1', 'pe

##### 2.2.2.2 Tokenize Data

Run **2.2.2.1 Harmonize Data** before this step.

In [ ]:
if folder_path or sample_key:
    dataset_list = []
    for adata_sample in adata_list:
        dataset_sample = tokenize_adata(
            adata_sample,
            model_folder_path,
            cache_directory_path,
            nproc=nproc,
            processing_mode=processing_mode)
        item = dataset_sample[0]
        print(item)
        dataset_list.append(dataset_sample)
        sample_name = adata_sample.obs[batch_key].unique().tolist()[0]
        adata_sample.write(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad") # write to disk to add neighbor graph
else:
    dataset = tokenize_adata(
        adata=adata,
        model_folder_path=model_folder_path,
        cache_directory_path=cache_directory_path,             
        nproc=nproc,
        processing_mode=processing_mode)
    item = dataset[0]
    print(item)
    adata.write(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")

In [ ]:
# Save Huggingface datasets for each sample
if folder_path or sample_key:
    for i, sample_name in enumerate(adata.obs[batch_key].unique().tolist()):
        dataset = dataset_list[i]
        dataset.save_to_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
                             num_shards=num_shards)
else:
    file_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = file_name.replace(".h5ad", "")
    dataset.save_to_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset",
                         num_shards=num_shards)

##### 2.2.2.3 Load Tokenized Data

In [17]:
# Load harmonized AnnData objects and Huggingface datasets for each sample
if folder_path:
    adata_list = []
    dataset_list = []
    for harmonized_adata_file_path in os.listdir(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}"):
        adata_sample = sc.read_h5ad(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{harmonized_adata_file_path}")
        dataset_sample = load_from_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{harmonized_adata_file_path.replace('.h5ad', '.dataset')}")
        adata_list.append(adata_sample)
        dataset_list.append(dataset_sample)
elif sample_key:
    adata_list = []
    dataset_list = []
    for i, sample_name in enumerate(adata.obs[batch_key].unique().tolist()):
        dataset_sample = load_from_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset")
        adata_sample = sc.read_h5ad(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
        adata_list.append(adata_sample)
        dataset_list.append(dataset_sample)
else:
    file_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = file_name.replace(".h5ad", "")
    adata = sc.read_h5ad(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.h5ad")
    dataset = load_from_disk(f"{tokenized_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}.dataset")

##### 2.2.2.4 Retrieve Embeddings

In [ ]:
if folder_path:
    for adata_sample, dataset_sample in zip(adata_list, dataset_list):

        sample_name = adata_sample.obs[batch_key].unique().tolist()[0]
        
        output_embed = embed_dataset(
            dataset=dataset_sample,
            model_folder_path=model_folder_path,
            emb_layer=emb_layer,
            agg_excluded_genes=agg_excluded_genes,
            top_k=top_k,
            batch_size=batch_size,
            pin_memory=pin_memory,
            num_workers=num_workers,
            ignore_spc_tokens=True)

        # Add embeddings to adata
        for key, values in output_embed.items():
            adata_sample.obsm[key] = values

        adata_sample.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}_{model_checkpoint}.h5ad")
elif sample_key:
    adata_emb_list = []
    for adata_sample, dataset_sample in zip(adata_list, dataset_list):

        sample_name = adata_sample.obs[batch_key].unique().tolist()[0]
        
        output_embed = embed_dataset(
            dataset=dataset_sample,
            model_folder_path=model_folder_path,
            emb_layer=emb_layer,
            agg_excluded_genes=agg_excluded_genes,
            top_k=top_k,
            batch_size=batch_size,
            pin_memory=pin_memory,
            num_workers=num_workers,
            ignore_spc_tokens=True)

        # Add embeddings to adata
        for key, values in output_embed.items():
            adata_sample.obsm[key] = values

        adata_emb_list.append(adata_sample)
        adata = ad.concat(adata_emb_list)
        adata.var['ensembl_id'] = adata_emb_list[0].var['ensembl_id']
        for key in adata_emb_list[0].obsp.keys():
            blocks = [a.obsp[key] for a in adata_emb_list]
            adata.obsp[key] = sp.block_diag(blocks, format="csr")
        file_name = file_path.split('/')[-1]
        adata.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{file_name.replace('.h5ad', f'_{model_checkpoint}.h5ad')}")    
else:
    file_name = file_path.split('/')[-1]
    sample_name = file_name.replace(".h5ad.gz", ".h5ad")
    sample_name = sample_name.replace(".h5ad", "")
    
    output_embed = embed_dataset(
        dataset=dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        agg_excluded_genes=agg_excluded_genes,
        top_k=top_k,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        ignore_spc_tokens=True)
    
    # Add embeddings to adata
    for key, values in output_embed.items():
        adata.obsm[key] = values

    adata.write(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}_{model_checkpoint}.h5ad")

### 2.3 (Optional) Analyze Raw Counts

#### 2.3.1 Load Harmonized AnnDatas

In [45]:
# Load AnnData with embeddings
if folder_path or sample_key:
    files = [os.path.join(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}", f) for f in os.listdir(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}") if f.endswith(".h5ad")]
    adata_list = []
    for f in files:
        print(f"Reading {f}")
        adata_file = ad.read_h5ad(f)
        adata_list.append(adata_file)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
    for key in adata_list[0].obsp.keys():
        blocks = [a.obsp[key] for a in adata_list]
        adata.obsp[key] = sp.block_diag(blocks, format="csr")
else:
    file_name = file_path.split('/')[-1]
    adata = sc.read_h5ad(f"{harmonized_data_folder_path}/{model_timestamp}/{project_name}/{file_name}")
print(adata)

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/terra_tutorial_artifacts/harmonized_data/TERRA-96M/my_terra_run'

In [ ]:
# Create dataset-specific label columns
if project_name == 'kidney_healthy_integration':
    adata.obs['assay'] = adata.obs['batch'].str.startswith(('DI', 'CV'), na=False) \
        .map({True: 'xenium', False: 'slideseqv2'})
    
    adata.obs['source'] = np.select(
        [
            adata.obs['batch'].str.startswith('DI', na=False),
            adata.obs['batch'].str.startswith('CV', na=False)
        ],
        ['DI', 'CV'],
        default='Puck'
    )

    label_keys = ['assay', 'source']
elif project_name == 'kidney_disease_integration_6k':
    adata.obs['assay'] = adata.obs['batch'].str.lower().str.startswith('cosmx', na=False) \
        .map({True: 'cosmx', False: 'xenium'})

    label_keys = ['total_counts', 'assay']
elif project_name == 'kidney_disease_integration':
    adata.obs['assay'] = adata.obs['batch'].str.lower().str.startswith('cosmx', na=False) \
        .map({True: 'cosmx', False: 'xenium'})

    bk = adata.obs[batch_key]
    
    adata.obs["source"] = np.select(
        [
            bk.str.startswith("DI"),
            bk.str.startswith("CV"),
            bk.str.contains("CosMx_Biopsy") | bk.str.contains("CosMx_Nephr"),
            bk.str.startswith("Control") | bk.str.startswith("SLE"),
            bk.str.contains("susztak"),
            bk.str.startswith("CosMx_Control") | bk.str.startswith("CosMx_UU"),
        ],
        [
            "xenium_DI",
            "xenium_CV",
            "edinburgh_cosmx_1k",
            "ln_cosmx",
            "susztak_cosmx",
            "edinburgh_cosmx_6k",
        ],
        default="UNKNOWN"
    )
    
    label_keys = ['total_counts', 'assay', 'source']
elif project_name == 'combined_pancreas':
    bk = adata.obs[batch_key]

    adata.obs["source"] = np.select(
        [
            bk.str.startswith(("Hml", "Sml")),
            bk.str.startswith("DJ"),
        ],
        [
            "fetal_pancreas",
            "adult_pancreas",
        ],
        default="UNKNOWN"
    )

    label_keys = ['source']
elif project_name == 'skin_eczema_multisample':
    label_keys = ['annotation_new7', 'NICHE_NAMES']
elif project_name == 'skin_eczema_folder':
    label_keys = ['annotation_new7', 'NICHE_NAMES']
elif project_name == 'skin_eczema_singlesample':
    label_keys = ['annotation_new7', 'NICHE_NAMES']

#### 2.3.2 Visualize PCA-based UMAP baseline

In [ ]:
# Plot cell-level expression-based UMAP
# Log-normalize counts
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4) # library size normalize
sc.pp.log1p(adata) # log-transform

try: # if rapids_singlecell is installed
    rsc.pp.pca(adata,
               n_comps=n_pca_pcs)
    
    rsc.pp.neighbors(adata,
                     n_neighbors=15,
                     use_rep="X_pca",
                     n_pcs=None,
                     key_added='expr')
    
    rsc.tl.umap(adata,
               neighbors_key='expr',
               key_added=f'expr_umap')
except NameError:
    sc.pp.pca(adata,
              n_comps=n_pca_pcs)
    
    sc.pp.neighbors(adata,
                    n_neighbors=15,
                    use_rep="X_pca",
                    n_pcs=None,
                    key_added='expr')
    
    sc.tl.umap(adata,
               neighbors_key='expr',
               key_added=f'expr_umap')
    
adata.obsm['X_umap'] = adata.obsm[f'expr_umap']

n_pca_pcs_string = f"_{n_pca_pcs}_pcs"
sc.pl.umap(
    adata,
    color=batch_key,
    show=False
)
plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_umap_by_{batch_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

for label_key in label_keys:
    sc.pl.umap(
        adata,
        color=label_key,
        show=False
    )
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_umap_by_{label_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

adata.X = adata.layers['counts'].copy()
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# Plot neighborhood-level expression-based UMAP
adata.layers["X_neigh"] = (
    adata.obsp["spatial_connectivities"].T @ adata.X)

# Log-normalize counts
#adata.layers["X_neigh_counts"] = adata.layers["X_neigh"].copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer="X_neigh")
sc.pp.log1p(adata, layer="X_neigh")

try: # if rapids_singlecell is installed
    sc.pp.pca(adata, # not enough GPU memory
              layer="X_neigh",
              n_comps=n_pca_pcs)
    
    rsc.pp.neighbors(adata,
        n_neighbors=15,
        use_rep="X_pca",
        n_pcs=None,
        key_added="expr_neigh")

    rsc.tl.umap(adata,
                neighbors_key="expr_neigh",
                key_added="expr_neigh_umap")
except NameError:
    sc.pp.pca(adata,
              layer="X_neigh",
              n_comps=n_pca_pcs)

    sc.pp.neighbors(adata,
                    n_neighbors=15,
                    use_rep="X_pca",
                    n_pcs=None,
                    key_added="expr_neigh")

    sc.tl.umap(adata,
               neighbors_key="expr_neigh",
               key_added="expr_neigh_umap")


adata.obsm["X_umap"] = adata.obsm["expr_neigh_umap"]

n_pca_pcs_string = f"_{n_pca_pcs}_pcs"
sc.pl.umap(adata, color=batch_key, show=False)
plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_neigh_umap_by_{batch_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

for label_key in label_keys:
    sc.pl.umap(adata, color=label_key, show=False)
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_neigh_umap_by_{label_key}{n_pca_pcs_string if n_pca_pcs else ''}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

#adata.layers["X_neigh"] = adata.layers["X_neigh_counts"].copy()
torch.cuda.empty_cache()
gc.collect()

#### 2.3.3 Visualize PCA-based Clustering

In [ ]:
# Plot cell-level expression-based clusters
latent_cluster_key = f"expr_clusters_res_{latent_leiden_resolution}"

try:
    rsc.tl.leiden(adata,
                  neighbors_key='expr',
                  key_added=latent_cluster_key,
                  resolution=latent_leiden_resolution,
                  flavor="igraph",
                  n_iterations=2)
except NameError:
    sc.tl.leiden(adata,
                 neighbors_key='expr',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution,
                 flavor="igraph",
                 n_iterations=2)

adata.obsm["X_umap"] = adata.obsm["expr_umap"]

sc.pl.umap(adata, color=latent_cluster_key, show=False)
plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_umap_by_{latent_cluster_key}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

samples = adata.obs[batch_key].unique().tolist()
for sample in samples:
    adata_sample = adata[adata.obs[batch_key] == sample]
    sc.pl.spatial(adata_sample,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[sample],
                  save=f'tissue_by_{latent_cluster_key}.png')
    os.rename(f'{results_folder_path}/{project_name}/showtissue_by_{latent_cluster_key}.png',
              f'{results_folder_path}/{project_name}/tissue_{sample}_by_{latent_cluster_key}.png')

In [ ]:
# Plot neighborhood-level expression-based clusters
latent_cluster_key = f"expr_neigh_clusters_res_{latent_leiden_resolution}"

try:
    rsc.tl.leiden(adata,
                  neighbors_key='expr_neigh',
                  key_added=latent_cluster_key,
                  resolution=latent_leiden_resolution,
                  flavor="igraph",
                  n_iterations=2)
except NameError:
    sc.tl.leiden(adata,
                 neighbors_key='expr_neigh',
                 key_added=latent_cluster_key,
                 resolution=latent_leiden_resolution,
                 flavor="igraph",
                 n_iterations=2)

adata.obsm["X_umap"] = adata.obsm["expr_neigh_umap"]

sc.pl.umap(adata, color=latent_cluster_key, show=False)
plt.savefig(os.path.join(results_folder_path, f"{project_name}/expr_neigh_umap_by_{latent_cluster_key}.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close()

samples = adata.obs[batch_key].unique().tolist()
for sample in samples:
    adata_sample = adata[adata.obs[batch_key] == sample]
    sc.pl.spatial(adata_sample,
                  color=latent_cluster_key,
                  palette=custom_palette,
                  spot_size=spot_sizes[sample],
                  save=f'tissue_by_{latent_cluster_key}.png')
    os.rename(f'{results_folder_path}/{project_name}/showtissue_by_{latent_cluster_key}.png',
              f'{results_folder_path}/{project_name}/tissue_{sample}_by_{latent_cluster_key}.png')

#### 2.3.4 Summarize Gene Expression

In [ ]:
combined_counts, top_gene_df = summarize_gene_expression(adata, list(adata.obs['cell_id']), top_n=1000)
display(combined_counts)
display(top_gene_df)

### 2.4 (Optional) Retrieve Average Gene Embeddings
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [18]:
print(len(list(adata.var['ensembl_id'])))

4949


In [19]:
cell_gene_ensembl_id = list(adata.var['ensembl_id'])
neighborhood_gene_ensembl_id = list(adata.var['ensembl_id'])

#cell_gene_ensembl_id = list(top_gene_df[:1000]['ensembl_id'])
#neighborhood_gene_ensembl_id = list(top_gene_df[:1000]['ensembl_id'])

In [20]:
dataset_list

[Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 339874
 }),
 Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 294940
 }),
 Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 449201
 }),
 Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 263360
 }),
 Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 509425
 }),
 Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 383108
 }),
 Dataset({
     features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
     num_rows: 3566

In [21]:
dataset = concatenate_datasets(dataset_list)

If you only want to keep specific cell IDs

In [30]:
dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 4393579
})

In [36]:
dataset['cell_id']

Column(['aaaaggln-1', 'aaaaglmo-1', 'aaaaiabe-1', 'aaaaihfg-1', 'aaabdfbd-1'])

In [40]:
cell_ids = pd.read_csv("/path/to/cell_ids.csv")

keep_ids = set(cell_ids.iloc[:, 0].astype(str).apply(lambda x: x.split('_')[0]))

filtered_dataset = dataset.filter(
    lambda example: str(example["cell_id"]) in keep_ids,
    num_proc=1
)

Filter: 100%|███████████████████████████████████████████████████████████████████████████████████| 4393579/4393579 [01:31<00:00, 47796.06 examples/s]


In [41]:
filtered_dataset

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'gene_tokens', 'gene_expr', 'n_nonzero_tokens'],
    num_rows: 164155
})

In [42]:
dataset = filtered_dataset

In [43]:
f"{embedded_data_folder_path}/{model_timestamp}/{project_name}"

'/path/to/terra_tutorial_artifacts/embedded_data/TERRA-96M/my_terra_run'

In [ ]:
for i, sample_name in enumerate(samples):
    if i == 0:
        continue
    dataset = dataset_list[i]

    output_average_gene_embed = get_average_gene_embed(
        dataset=dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}_{model_checkpoint}_full_genes_endothelial_barcodes_embryo_avg_gene_embed.pkl", "wb") as f:
        pickle.dump(output_average_gene_embed, f)

STEP 1: LOADING CONFIG...
STEP 2: GETTING AVERAGE GENE EMBEDDINGS


INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(13, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

Special token pad ratio: 1.0


INFO:root:Dataloader and -sampler created.
INFO:root:Loaded pretrained target encoder from epoch 1 with msg: <All keys matched successfully>.
INFO:root:Finished loading checkpoint with read path: /path/to/model_checkpoint.pt.


['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr']


...

INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(13, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

Special token pad ratio: 1.0


INFO:root:Dataloader and -sampler created.
INFO:root:Loaded pretrained target encoder from epoch 1 with msg: <All keys matched successfully>.
INFO:root:Finished loading checkpoint with read path: /path/to/model_checkpoint.pt.


['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr']


6it [00:32,  5.20s/it]

In [ ]:
#adata_sample = adata[adata.obs[sample_key] == sample_name]

In [ ]:
f"{embedded_data_folder_path}/{model_timestamp}/{project_name}"

In [ ]:
for i, sample_name in enumerate(samples):
    if i == 0:
        continue
    dataset = dataset_list[i]

    output_average_gene_embed = get_average_gene_embed(
        dataset=dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{sample_name}_{model_checkpoint}_full_genes_avg_gene_embed.pkl", "wb") as f:
        pickle.dump(output_average_gene_embed, f)

In [ ]:
f"{embedded_data_folder_path}/{model_timestamp}/{project_name}"

In [ ]:
adata_sample.uns['cell_gene_emb_average_per_data'] = output_average_gene_embed['cell_gene_emb_average_per_data']
adata_sample.uns['cell_gene_emb_counts_per_data'] = output_average_gene_embed['cell_gene_emb_counts_per_data']
adata_sample.uns['neighborhood_gene_emb_average_per_data'] = output_average_gene_embed['neighborhood_gene_emb_average_per_data']
adata_sample.uns['neighborhood_gene_emb_counts_per_data'] = output_average_gene_embed['neighborhood_gene_emb_counts_per_data']

In [ ]:
sample_name

In [ ]:
adata_sample.write()

In [ ]:
# 1) You can choose any of the embeddings, which could be either a cell or a neighborhood.
gene_emb_name = 'neighborhood_gene_emb_average_per_data'
gene_emb_name = 'cell_gene_emb_average_per_data'


In [ ]:
gene_emb_name

In [ ]:
gene_emb = np.array(adata.uns[gene_emb_name])  # (n_genes, emb_dim)
gene_ids =  np.array(cell_gene_ensembl_id,  dtype=str)                  # (n_genes,)

# 2) Remove any gene rows containing NaNs
mask = ~np.isnan(gene_emb).any(axis=1)
gene_emb_clean = gene_emb[mask]
gene_ids_clean = gene_ids[mask]

# 3) Build a small AnnData where each "obs" is one gene
adata_genes = ad.AnnData(
    X=gene_emb_clean,
    obs=pd.DataFrame({'gene_id': gene_ids_clean})
)
adata_genes.obs_names = gene_ids_clean.astype(str)

# 4) Compute a neighborhood graph on the embeddings
sc.pp.neighbors(
    adata_genes,
    use_rep='X',
    n_neighbors=15,
    metric='euclidean'
)

# 5) Run UMAP
sc.tl.umap(
    adata_genes,
    #min_dist=0.5,
    #spread=1.0
)


# 7) (Optional) Quick plot
sc.pl.umap(
    adata_genes,
    title='Gene embedding UMAP',
    show=True
)


In [ ]:
sc.tl.pca(adata_genes, n_comps=2, use_highly_variable=False, svd_solver='arpack')

sc.pl.pca(
    adata_genes,
    components=['1,2'],   # plot PC1 vs PC2
    color=None,
    title='Gene embeddings PCA (PC1 vs PC2)',
    show=True
)


In [ ]:
# Store results here
results = []

# Define parameters
knn_values = [5, 10, 15, 20]
resolutions = np.arange(0.1, 1.1, 0.1)  # 0.1 to 1.0

# Loop over each KNN and resolution combination
for knn in knn_values:
    sc.pp.neighbors(adata_genes, n_neighbors=knn, use_rep='X')  # use 'X' or other embedding like 'X_pca'
    
    for res in resolutions:
        # Run Leiden clustering
        sc.tl.leiden(adata_genes, resolution=res, key_added='leiden_tmp')
        
        # Compute silhouette score
        labels = adata_genes.obs['leiden_tmp']
        if len(np.unique(labels)) < 2:
            score = np.nan  # silhouette score not defined for 1 cluster
        else:
            score = silhouette_score(adata_genes.obsm['X_pca'], labels)  # you can change to 'X' if no PCA
        
        # Store results
        results.append({
            'knn': knn,
            'resolution': round(res, 2),
            'silhouette_score': score
        })

# Convert results to DataFrame
df = pd.DataFrame(results)
df['combo'] = df['knn'].astype(str) + '_k_' + df['resolution'].astype(str)

# Plot
plt.figure(figsize=(12, 6))
df_sorted = df.sort_values('silhouette_score', ascending=False)
plt.bar(df_sorted['combo'], df_sorted['silhouette_score'], color='skyblue')
plt.xticks(rotation=90)
plt.ylabel("Silhouette Score")
plt.xlabel("KNN_Resolution Combination")
plt.title("Silhouette Score for Different KNN and Leiden Resolution Settings")
plt.tight_layout()
plt.show()


In [ ]:
sc.pp.neighbors(adata_genes,n_neighbors=20)
sc.tl.leiden(adata_genes,resolution=0.3,key_added='r9')

In [ ]:
sc.pl.umap(adata_genes,color='r9',frameon=False)

In [ ]:
id_to_name = {ensg: name for name, ensg in adata.var['ensembl_id'].items()}

In [ ]:
interesting_group = '10'
ens_interesting_group = list(adata_genes.obs[adata_genes.obs['r9']==interesting_group]['gene_id'])

In [ ]:
genes  = [ id_to_name.get(ensg, "UNKNOWN") for ensg in ens_interesting_group]

In [ ]:
len(genes)

In [ ]:
# 2) Your input gene list
# 3) Call gseapy.enrichr
#    You can choose any Enrichr library; here are some examples:
# I should create a mapping from gene_id to ens
# from 110 release
libraries = [
    'KEGG_2019_Human',
    'GO_Biological_Process_2021',
    'Reactome_Pathways_2024',
    'MSigDB_Hallmark_2020'
]

# Run ORA for each library
results = {}
for lib in libraries:
    enr = gp.enrichr(
        gene_list = genes,
        gene_sets  = lib,
        outdir     = None,        # don't write files
        cutoff     = 0.05,         # only show terms with adjusted p < 0.05
    )
    # enr.results is a pandas DataFrame
    results[lib] = enr.results

# 4) Inspect top enriched terms for each library
for lib, df in results.items():
    print(f"\n=== Top 5 terms in {lib} ===")
    display(df[['Term','Adjusted P-value','Combined Score']].head(20))


### 2.5 (Optional) Retrieve Cell-level Gene Embeddings

In [ ]:
adata.var

In [ ]:
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

**TODO**: Specify the gene ensembl IDs for which you want to retrieve cell-level gene embeddings. 

In [ ]:
cell_gene_ensembl_id = ['ENSG00000104419', 'ENSG00000130402']
neighborhood_gene_ensembl_id = ['ENSG00000104419','ENSG00000130402'] # ['all'] * len(perturbed_cell_id)

In [ ]:
output_gene_embed = get_gene_embed( dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
for gene_ens in output_gene_embed.keys():
    adata.obsm[gene_ens] = output_gene_embed[gene_ens]

In [ ]:
output_gene_embed

In [ ]:
A = adata.obsm['neighborhood_emb_geneENSG00000104419']
B = adata.obsm['neighborhood_emb_geneENSG00000130402']

# Compute row-wise norms
norm_A = np.linalg.norm(A, axis=1)
norm_B = np.linalg.norm(B, axis=1)

# Mask: keep only rows where BOTH are non-zero
mask = (norm_A > 0) & (norm_B > 0)

# Compute cosine similarity only for valid rows
cos_sim = np.sum(A[mask] * B[mask], axis=1) / (norm_A[mask] * norm_B[mask])

print(cos_sim)

In [ ]:
def compute_mean_unmasked_emb(emb: torch.Tensor,
                              mask: torch.Tensor,
                              ) -> torch.Tensor:
    """
    Compute the mean of unmasked embedding positions.
    
    Parameters
    -----------
    emb:
        The input embeddings tensor (3D).
    mask:
        A 2D boolean mask tensor indicating the sequence positions that mean
        should be computed over.
    
    Returns
    -----------
    mean_emb:
        The mean embedding tensor.

    Raises
    -----------
    ValueError: If the emb tensor is not 3D.
    """
    # Use broadcasting to sum embeddings across unmasked positions
    if emb.dim() == 3:
        # If the embeddings tensor has 3 dimensions (batch_size,
        # sequence_length, embedding_dim), broadcast the mask to match the
        # dimensions of emb. The mask tensor is initially (batch_size,
        # sequence_length), so we unsqueeze to (batch_size, sequence_length, 1)
        masked_emb = emb * mask.unsqueeze(2) # broadcast the mask along the
                                             # embedding dimension

        # Sum the masked embeddings along the sequence dimension
        sum_emb = masked_emb.sum(1)

        # Calculate the mean by dividing the summed embeddings by the number of
        # unmasked positions. The mask is summed to count unmasked tokens, and
        # view(-1, 1) ensures the resulting tensor has the correct dimensions
        # for broadcasting during division. The + 1e-9 will handle the case
        # where we are retrieving a gene that may have
        # mask.sum(dim).view(-1, 1).float() = 0
        mean_emb = sum_emb / (mask.sum(1).view(-1, 1).float() + 1e-9)

    else:
        raise ValueError('Expected a 3D tensor for emb, but got a tensor with'
                         f'{emb.dim()} dimensions.')

    return mean_emb

In [ ]:
A[mask]

In [ ]:
B[mask]

In [ ]:
mask

In [ ]:
cos_sim.shape

In [ ]:
np.mean(cos_sim)

### 2.6 Analyze Embeddings

#### 2.6.1 Load Embeddings

In [ ]:
# Load AnnData with embeddings
if folder_path:
    files = [os.path.join(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}", f) for f in os.listdir(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}") if f.endswith(f"_{model_checkpoint}.h5ad")]
    adata_list = []
    for f in files:
        print(f"Reading {f}")
        adata_file = ad.read_h5ad(f)
        adata_list.append(adata_file)
    adata = ad.concat(adata_list)
    adata.var['ensembl_id'] = adata_list[0].var['ensembl_id']
    for key in adata_list[0].obsp.keys():
        blocks = [a.obsp[key] for a in adata_list]
        adata.obsp[key] = sp.block_diag(blocks, format="csr")
else:
    file_name = file_path.split('/')[-1]
    adata = sc.read_h5ad(f"{embedded_data_folder_path}/{model_timestamp}/{project_name}/{file_name.replace('.h5ad', f'_{model_checkpoint}.h5ad')}")
print(adata)

In [ ]:
# Create dataset-specific label columns
if project_name == 'kidney_healthy_integration':
    adata.obs['assay'] = adata.obs['batch'].str.startswith(('DI', 'CV'), na=False) \
        .map({True: 'xenium', False: 'slideseqv2'})
    
    adata.obs['source'] = np.select(
        [
            adata.obs['batch'].str.startswith('DI', na=False),
            adata.obs['batch'].str.startswith('CV', na=False)
        ],
        ['DI', 'CV'],
        default='Puck'
    )

    label_keys = ['assay', 'source']
elif project_name == 'kidney_disease_integration_6k':
    adata.obs['assay'] = adata.obs['batch'].str.lower().str.startswith('cosmx', na=False) \
        .map({True: 'cosmx', False: 'xenium'})

    label_keys = ['total_counts', 'assay']
elif project_name == 'kidney_disease_integration':
    adata.obs['assay'] = adata.obs['batch'].str.lower().str.startswith('cosmx', na=False) \
        .map({True: 'cosmx', False: 'xenium'})

    bk = adata.obs[batch_key]
    
    adata.obs["source"] = np.select(
        [
            bk.str.startswith("DI"),
            bk.str.startswith("CV"),
            bk.str.contains("CosMx_Biopsy") | bk.str.contains("CosMx_Nephr"),
            bk.str.startswith("Control") | bk.str.startswith("SLE"),
            bk.str.contains("susztak"),
            bk.str.startswith("CosMx_Control") | bk.str.startswith("CosMx_UU"),
        ],
        [
            "xenium_DI",
            "xenium_CV",
            "edinburgh_cosmx_1k",
            "ln_cosmx",
            "susztak_cosmx",
            "edinburgh_cosmx_6k",
        ],
        default="UNKNOWN"
    )
    
    label_keys = ['total_counts', 'assay', 'source']
elif project_name == 'combined_pancreas':
    bk = adata.obs[batch_key]

    adata.obs["source"] = np.select(
        [
            bk.str.startswith(("Hml", "Sml")),
            bk.str.startswith("DJ"),
        ],
        [
            "fetal_pancreas",
            "adult_pancreas",
        ],
        default="UNKNOWN"
    )

    label_keys = ['source']
elif project_name == 'skin_eczema_multisample':
    label_keys = ['annotation_new7', 'NICHE_NAMES']
elif project_name == 'skin_eczema_folder':
    label_keys = ['annotation_new7', 'NICHE_NAMES']
elif project_name == 'skin_eczema_singlesample':
    label_keys = ['annotation_new7', 'NICHE_NAMES']
elif project_name == 'mmb0-1b_smb1-1b_1p':
    label_keys = ['nichecompass_latent_cluster', 'cell_type']

#### 2.6.2 Validate & Visualize Embeddings

In [ ]:
# Compute average cosine similarity to check embeddings are not collapsed
# (this should be < 0.9 otherwise the embeddings might not be useful; this can happen if the gene panel is too small)
for emb_key in ['neighborhood_emb', 'cell_emb']:
    X = adata.obsm[emb_key]
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
    n = X.shape[0]
    S = X.sum(axis=0)
    avg_cos_sim = (S @ S - n) / (n * (n - 1))
    print(f'{emb_key} average cosine similarity: {avg_cos_sim}')

In [ ]:
# Compute average cell-wise cosine similarity between embeddings
for k1, k2 in combinations(['neighborhood_emb', 'cell_emb'], 2):
    A = adata.obsm[k1]
    B = adata.obsm[k2]

    # Normalize rows
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)

    # Row-wise cosine similarity
    cos_sim = np.sum(A * B, axis=1)

    print(f'{k1} vs {k2} mean cosine similarity: {cos_sim.mean()}')

In [ ]:
# Plot embedding-based UMAPs
for emb_key in ['neighborhood_emb', 'cell_emb']:
    try: # if rapids_singlecell is installed
        rsc.pp.neighbors(adata,
                         n_neighbors=15,
                         use_rep=emb_key,
                         n_pcs=n_pcs,
                         key_added=emb_key)
        
        rsc.tl.umap(adata,
                    neighbors_key=emb_key,
                    key_added=f'{emb_key}_umap')
    except NameError:
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=emb_key,
                        n_pcs=n_pcs,
                        key_added=emb_key)
        
        sc.tl.umap(adata,
                   neighbors_key=emb_key,
                   key_added=f'{emb_key}_umap')
    
    adata.obsm["X_umap"] = adata.obsm[f"{emb_key}_umap"]

    n_pcs_string = f"_{n_pcs}_pcs"
    sc.pl.umap(adata, color=batch_key, show=False)
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/{model_checkpoint}_{emb_key}_umap_by_{batch_key}{n_pcs_string if n_pcs else ''}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    for label_key in label_keys:
        sc.pl.umap(adata, color=label_key, show=False)
        plt.savefig(os.path.join(results_folder_path, f"{project_name}/{model_checkpoint}_{emb_key}_umap_by_{label_key}{n_pcs_string if n_pcs else ''}.png"), dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

In [ ]:
# Save intermediary results
adata.write(os.path.join(results_folder_path, f"{project_name}/{model_checkpoint}_adata_processed.h5ad"))

#### 2.6.3 Niche & Cell Type Identification

In [ ]:
# Plot embedding-based clusters
for emb_key in ['neighborhood_emb', 'cell_emb']:
    latent_cluster_key = f"{emb_key}_clusters_res_{latent_leiden_resolution}"
    
    try:
        rsc.tl.leiden(adata,
                      neighbors_key=emb_key,
                      key_added=latent_cluster_key,
                      resolution=latent_leiden_resolution,
                      flavor="igraph",
                      n_iterations=2)
    except NameError:
        sc.tl.leiden(adata,
                     neighbors_key=emb_key,
                     key_added=latent_cluster_key,
                     resolution=latent_leiden_resolution,
                     flavor="igraph",
                     n_iterations=2)
    
    adata.obsm["X_umap"] = adata.obsm[f"{emb_key}_umap"]
    
    sc.pl.umap(adata, color=latent_cluster_key, show=False)
    plt.savefig(os.path.join(results_folder_path, f"{project_name}/{model_checkpoint}_{emb_key}_umap_by_{latent_cluster_key}.png"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    
    samples = adata.obs[batch_key].unique().tolist()
    for sample in samples:
        adata_sample = adata[adata.obs[batch_key] == sample]
        sc.pl.spatial(adata_sample,
                      color=latent_cluster_key,
                      palette=custom_palette,
                      spot_size=spot_sizes[sample],
                      save=f'tissue_by_{latent_cluster_key}.png')
        os.rename(f'{results_folder_path}/{project_name}/showtissue_by_{latent_cluster_key}.png',
                  f'{results_folder_path}/{project_name}/{model_checkpoint}_tissue_{sample}_by_{latent_cluster_key}.png')

In [ ]:
# Save intermediary results
adata.write(os.path.join(results_folder_path, f"{project_name}/adata_processed.h5ad"))

In [ ]:
os.path.join(results_folder_path, f"{project_name}/adata_processed.h5ad")

#### 2.6.4 Compute EMD Distance

In [ ]:
# each list should at least contain 20 gene each 
# to have score that is meaningful
# compute mmd also and add it 
cell_gene_ensembl_id = list(top_gene_df['ensembl_id'])
neighborhood_gene_ensembl_id = list(top_gene_df['ensembl_id'])  # ['all'] * len(perturbed_cell_id)


In [ ]:
output_emd_distance = get_emd_distance( dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
adata.obs['emd_dist'] = output_emd_distance

In [ ]:
# 2) Normalize your data so that 0→0; 0.6→1.0
norm = Normalize(vmin=0.0, vmax=max(output_emd_distance))

# 3) Build the colormap
#cmap = make_thresholded_cmap(base_cmap="plasma", threshold=0.8)

sq.pl.spatial_scatter(
        adata,
        color='emd_dist',
        #cmap=cmap,
        norm=norm,
        shape=None,
        figsize=(10, 8),
        size=1.5,
        title=f"{dataset_name} - (EMD distance)",
)
plt.show()

#### 2.6.5 Compute Spatial Score
Please note that the model was trained using Ensembl release 110, so for correct mapping you should use release 110 when looking up a gene’s Ensembl ID.

In [ ]:
adata.var

In [ ]:
# for out put of get_top_gene_score be correct both cell_gene_ensembl_id and 
# neighborhood_gene_ensembl_id should have same size and same order
cell_gene_ensembl_id =  list(top_gene_df['ensembl_id']) 
neighborhood_gene_ensembl_id =  list(top_gene_df['ensembl_id'])  # ['all'] * len(perturbed_cell_id)

In [ ]:
output_cosine_sim_score = get_spatial_score(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
gene_pair_score = output_cosine_sim_score['cos_sim_neighborhood'] / output_cosine_sim_score['cos_sim_cell']

In [ ]:
gene_pair_score

In [ ]:
output_cosine_sim_score['cell_count_neighborhood']

In [ ]:
df_gene_scores = get_top_gene_score(gene_pair_score, 
                                    cell_gene_ensembl_id, 
                                    gene_df=adata.var, 
                                    gene_counts= output_cosine_sim_score['cell_count_neighborhood'],
                                    min_count = 20)

In [ ]:
df_gene_scores.sort_values(by='gene_score', ascending=False)

In [ ]:
df_gene_pairs_score = get_top_gene_pairs(
    torch.from_numpy(gene_pair_score),
    count_cell=torch.from_numpy(output_cosine_sim_score['cell_count_neighborhood']),
    count_neb=torch.from_numpy(output_cosine_sim_score['cell_count_cell']),
    cos_sim_cell=torch.from_numpy(output_cosine_sim_score['cell_count_cell']),
    cos_sim_neb=torch.from_numpy(output_cosine_sim_score['cos_sim_neighborhood']),
    gene_df=adata.var,
    cell_gene_ids=cell_gene_ensembl_id,
    neighborhood_gene_ids=neighborhood_gene_ensembl_id,
    min_count=20,
    sim_thresh=0.0,
    k=500
)


In [ ]:
df_gene_pairs_score

### 2.7 Analyze Subsetted Data

Here we show how to subset your dataset to a specific set of cells using cell IDs (e.g. specific niche, cell type, ...). You can then repeat all the analysis from 2.6 on this subset of the data.

In [ ]:
subset_cell_ids = ['1002_batch40_0',
 '1002_batch40_1',
 '1002_batch40_2',
 '1002_batch40_3',
 '1002_batch40_4',
 '1002_batch40_5',
 '1002_batch40_6',
 '1002_batch40_7',
 '1002_batch40_8',
 '1002_batch40_9']

In [ ]:
dataset_subset = subset_by_cell_ids(dataset, subset_cell_ids)
dataset_subset

### 2.8 Perform Perturbations

#### 2.1.13 Perturbed Data

In [ ]:
# Define perturbations
perturbed_cell_id = ['1007_batch20_0', '1007_batch20_0', '1007_batch20_1']
perturbed_ensembl_id = ['all', 'all', 'ENSG00000169194'] # ['all'] * len(perturbed_cell_id)
perturbation_target = ['cell', 'neighborhood', 'cell'] # ['neighborhood'] * len(perturbed_cell_id)
perturbation_type = ['knockout', 'knockout', 'foldchange'] # ['knockout'] * len(perturbed_cell_id)
foldchange = [np.nan, np.nan, 0.5] # [np.nan] * len(perturbed_cell_id)

perturb_df =  pd.DataFrame({
    'perturbed_cell_id': perturbed_cell_id,
    'perturbed_ensembl_id': perturbed_ensembl_id,
    'perturbation_target': perturbation_target,
    'perturbation_type': perturbation_type, 
    'foldchange': foldchange
})

In [ ]:
perturbed_dataset = perturb_dataset(
    dataset=dataset,
    perturb_df=perturb_df,
    model_folder_path=model_folder_path)

#### 2.1.14 Embed Perturb Data

In [ ]:
perturbed_output_embed = embed_dataset(
    dataset=perturbed_dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    cell_gene_ids=cell_gene_ids,
    neighborhood_gene_ids=neighborhood_gene_ids,
    agg_excluded_tokens=agg_excluded_tokens,
    top_k=top_k,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers)

In [ ]:
adata.obsm['perturbed_cell_emb'] = perturbed_output_embed['cell_emb']
adata.obsm['perturbed_neighborhood_emb'] = perturbed_output_embed['neighborhood_emb']

In [ ]:
label = cell_type_key

# Neighborhood embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_neighborhood_emb',
                key_added='perturbed_neighborhood')
sc.tl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           key_added='perturbed_neighborhood_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_neighborhood_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_neighborhood',
           color=label)

# Cell embedding UMAP
sc.pp.neighbors(adata,
                n_neighbors=15,
                use_rep='perturbed_cell_emb',
                key_added='perturbed_cell')
sc.tl.umap(adata,
           neighbors_key='perturbed_cell',
           key_added='perturbed_cell_umap')
adata.obsm['X_umap'] = adata.obsm['perturbed_cell_umap']
sc.pl.umap(adata,
           neighbors_key='perturbed_cell',
           color=label)